# Walmart Store Sales Forecasting — PatchTST

**ლოგირება:** Weights & Biases — პროექტი `ML-Final`, group `PatchTST_Training`.

**რა არის PatchTST** (Nie et al., *"A Time Series is Worth 64 Words"*, ICLR 2023):
სერია იყოფა non-overlapping ან overlapping **patch**-ებად (დროის ღერძის "სიტყვები"), რომლებზეც სტანდარტული Transformer encoder მუშაობს — ეს ამცირებს token-ების რაოდენობას long-sequence-ებზე და საშუალებას აძლევს attention-ს ლოკალურ semantic ინფორმაციაზე ფოკუსირდეს, თითო წერტილის ნაცვლად. Channel-independent არქიტექტურაა (თითო სერია calan calibrated ცალკე გადის encoder-ში, წონები საერთოა).

Run-ების სტრუქტურა იგივეა, რაც N-BEATS-ში — Preprocessing, candidate sweep, Final Pipeline. IsHoliday ეგზოგენური ფიჩერი აქაც გამოიყენება.

In [1]:
!pip install -q -U "sympy>=1.13.3"
!pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 38.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import wandb

wandb.login()

WANDB_PROJECT = 'ML-Final'
GROUP = 'PatchTST_Training'
NB_VERSION = 'v1'

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: amakh23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']
VAL_CUTOFF = '2012-08-17'


def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## Run 1 — Preprocessing

იგივე Date × (Store, Dept) მატრიცა, რაც DLinear-ში და NBEATS-ში. **მნიშვნელოვანი:** ეს pivot ავტომატურად ინახავს ყველა სერიას სრული სიგრძით (ადრეული/მოკლე სერიები 0-ით ივსება), ამიტომ short-series-ის ცალკე გაფილტვრა საჭირო არაა — არც ერთი Store-Dept წყვილი არ იკარგება.

დამატებით ვაშენებთ **holiday ვექტორს** — Date -> IsHoliday (ცნობილი მომავალი ეგზოგენური ფიჩერი, ერთნაირია ყველა store-ისთვის მოცემულ თარიღზე).

In [5]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='PatchTST_Preprocessing',
                 job_type='preprocessing',
                 config={'negative_sales_strategy': 'clip_to_zero',
                         'missing_strategy': 'fill_zero (late-starting / gappy series)',
                         'normalization': 'per-series standardization, std clipped to >= 1',
                         'exogenous': 'IsHoliday (known future covariate)'})

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

wide = train.pivot_table(index='Date', columns=['Store', 'Dept'], values='Weekly_Sales')
wide = wide.sort_index()
wide.index.name = 'Date'

missing_pct = float(wide.isna().mean().mean())
wide = wide.fillna(0.0)

holiday_by_date = train.groupby('Date')['IsHoliday'].max()
full_date_range = pd.DatetimeIndex(sorted(set(wide.index) | set(test['Date'].unique())))
holiday_by_date = holiday_by_date.reindex(full_date_range).fillna(False).astype(bool)

run.config['matrix_shape'] = str(wide.shape)
run.config['n_series_kept'] = wide.shape[1]
run.log({'missing_cell_pct': missing_pct, 'n_series': wide.shape[1], 'n_weeks': wide.shape[0]})
print('weeks x series:', wide.shape, '| missing cells:', f'{missing_pct:.1%}', '| n_series kept:', wide.shape[1])
run.finish()

val_dates = wide.index[wide.index >= VAL_CUTOFF]
PRED_LEN_VAL = len(val_dates)
print('validation weeks:', PRED_LEN_VAL)

weeks x series: (143, 3331) | missing cells: 11.5% | n_series kept: 3331


/tmp/ipykernel_3460/4227977353.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  holiday_by_date = holiday_by_date.reindex(full_date_range).fillna(False).astype(bool)


missing_cell_pct,▁
n_series,▁
n_weeks,▁
missing_cell_pct,0.11497
n_series,3331
n_weeks,143


validation weeks: 11


## PatchTST იმპლემენტაცია (PyTorch, გამარტივებული + ეგზოგენური holiday input)

- სერია იყოფა patch-ებად, თითოეული linear-embed-დება
- სტანდარტული Transformer encoder (multi-head self-attention + FFN)
- საბოლოო flatten + linear head პროგნოზირებს `pred_len`-ს
- **channel-shared** (ისევე, როგორც N-BEATS/DLinear-ში — ერთი წონების ნაკრები ყველა სერიაზე)
- IsHoliday მომავალი ჰორიზონტისთვის ემატება encoder-ის output-ს linear head-ის წინ

In [6]:
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print('device:', DEVICE)


class PatchTST(nn.Module):
    def __init__(self, seq_len, pred_len, patch_len=16, stride=8, d_model=64,
                 n_heads=4, n_layers=2, exog_size=0):
        super().__init__()
        self.seq_len, self.pred_len = seq_len, pred_len
        self.patch_len, self.stride = patch_len, stride
        self.n_patches = (seq_len - patch_len) // stride + 1

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        head_in = self.n_patches * d_model + exog_size
        self.head = nn.Linear(head_in, pred_len)

    def forward(self, x, exog=None):  # x: (B, seq_len)
        B = x.shape[0]
        patches = x.unfold(dimension=1, size=self.patch_len, step=self.stride)  # (B, n_patches, patch_len)
        h = self.patch_embed(patches) + self.pos_embed  # (B, n_patches, d_model)
        h = self.encoder(h)  # (B, n_patches, d_model)
        h = h.reshape(B, -1)  # (B, n_patches * d_model)
        if exog is not None:
            h = torch.cat([h, exog], dim=-1)
        return self.head(h)

device: cuda


In [7]:
from torch.utils.data import DataLoader, TensorDataset


def make_windows(arr, holiday_arr, seq_len, pred_len):
    xs, ys, es = [], [], []
    for t in range(arr.shape[0] - seq_len - pred_len + 1):
        xs.append(arr[t:t + seq_len])
        ys.append(arr[t + seq_len:t + seq_len + pred_len])
        es.append(holiday_arr[t + seq_len:t + seq_len + pred_len].astype(np.float32))
    return np.stack(xs), np.stack(ys), np.stack(es)


def train_patchtst(arr, holiday_arr, seq_len, pred_len, patch_len=16, stride=8,
                    d_model=64, n_heads=4, n_layers=2, epochs=80, batch_size=16,
                    lr=1e-3, log_fn=None):
    X, Y, E = make_windows(arr, holiday_arr, seq_len, pred_len)
    N = arr.shape[1]
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                        torch.tensor(Y, dtype=torch.float32),
                        torch.tensor(E, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = PatchTST(seq_len, pred_len, patch_len, stride, d_model, n_heads, n_layers,
                      exog_size=pred_len).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train()
        tot = 0.0
        for xb, yb, eb in dl:
            B = xb.shape[0]
            xb = xb.permute(0, 2, 1).reshape(B * N, seq_len).to(DEVICE)
            yb = yb.permute(0, 2, 1).reshape(B * N, pred_len).to(DEVICE)
            eb = eb.unsqueeze(1).repeat(1, N, 1).reshape(B * N, pred_len).to(DEVICE)
            opt.zero_grad()
            pred = model(xb, eb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            tot += loss.item() * B
        losses.append(tot / len(ds))
        if log_fn:
            log_fn(ep, losses[-1])
        if (ep + 1) % 10 == 0:
            print(f'  epoch {ep + 1:>3}: train MSE {losses[-1]:.4f}')
    return model, losses


def forecast_next(model, arr, exog_future, seq_len):
    model.eval()
    N = arr.shape[1]
    x = torch.tensor(arr[-seq_len:], dtype=torch.float32).permute(1, 0).to(DEVICE)
    exog = torch.tensor(exog_future, dtype=torch.float32).unsqueeze(0).repeat(N, 1).to(DEVICE)
    with torch.no_grad():
        pred = model(x, exog)
    return pred.cpu().numpy().T


val_actual = train[train['Date'] >= VAL_CUTOFF][['Store', 'Dept', 'Date', 'IsHoliday', 'Weekly_Sales']]


def matrix_val_wmae(pred_matrix):
    pred_df = pd.DataFrame(pred_matrix, index=val_dates, columns=wide.columns)
    pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()
    pred_long.columns = ['Date', 'Store', 'Dept', 'pred']
    merged = val_actual.merge(pred_long, on=['Store', 'Dept', 'Date'], how='left')
    merged['pred'] = merged['pred'].fillna(merged['Weekly_Sales'].mean())
    return wmae(merged['Weekly_Sales'], merged['pred'], merged['IsHoliday'])

In [12]:
tr_wide = wide[wide.index < VAL_CUTOFF]
mu = tr_wide.values.mean(axis=0)
sigma = np.clip(tr_wide.values.std(axis=0), 1.0, None)
norm_tr = (tr_wide.values - mu) / sigma

tr_holiday = holiday_by_date.reindex(tr_wide.index).values
val_holiday_future = holiday_by_date.reindex(val_dates).values.astype(np.float32)

CANDIDATES = [
    dict(seq_len=52, patch_len=16, stride=8, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=16, stride=8, d_model=32, n_heads=2, n_layers=1, epochs=80, lr=1e-3),
    dict(seq_len=78, patch_len=16, stride=8, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
]

best_val, BEST_CFG = np.inf, None
for i, cfg in enumerate(CANDIDATES):
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'PatchTST_candidate_{i}',
                     job_type='hyperparam_search',
                     config={**cfg, 'pred_len': PRED_LEN_VAL, 'exogenous': 'IsHoliday',
                             'val_scheme': f'train < {VAL_CUTOFF}, predict {PRED_LEN_VAL} weeks ahead'})
    print(f'candidate {i}: {cfg}')
    model, losses = train_patchtst(norm_tr, tr_holiday, pred_len=PRED_LEN_VAL, **cfg,
                                    log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))
    pred = forecast_next(model, norm_tr, val_holiday_future, cfg['seq_len']) * sigma + mu
    score = float(matrix_val_wmae(pred))
    run.summary['val_wmae'] = score
    run.finish()
    print(f'  -> val WMAE {score:,.2f}')
    if score < best_val:
        best_val, BEST_CFG = score, cfg

print('Best:', BEST_CFG, f'-> {best_val:,.2f}')

candidate 0: {'seq_len': 52, 'patch_len': 16, 'stride': 8, 'd_model': 64, 'n_heads': 4, 'n_layers': 2, 'epochs': 80, 'lr': 0.001}


AttributeError: module 'sympy' has no attribute 'printing'

## Run 3 — Final Pipeline

საბოლოო მოდელი მთელ ისტორიაზე ტრენინგდება და Kaggle test-ის 39 კვირას პროგნოზირებს, რეალური test.csv-ის holiday ვექტორის გამოყენებით.

In [11]:
import cloudpickle

test_dates = pd.DatetimeIndex(np.sort(test['Date'].unique()), name='Date')
PRED_LEN_TEST = len(test_dates)
assert (test_dates[0] - wide.index.max()).days == 7, 'test must start right after train'
test_holiday_future = holiday_by_date.reindex(test_dates).values.astype(np.float32)
print('test horizon:', PRED_LEN_TEST, 'weeks')


class PatchTSTForecastPipeline:
    def __init__(self, forecast_long, sd_mean, dept_mean, global_mean):
        self.forecast_long = forecast_long
        self.sd_mean = sd_mean
        self.dept_mean = dept_mean
        self.global_mean = global_mean

    def predict(self, model_input):
        df = model_input.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        out = (df.merge(self.forecast_long, on=['Store', 'Dept', 'Date'], how='left')
                 .merge(self.sd_mean, on=['Store', 'Dept'], how='left')
                 .merge(self.dept_mean, on='Dept', how='left'))
        pred = (out['pred'].fillna(out['SD_Mean'])
                           .fillna(out['Dept_Mean'])
                           .fillna(self.global_mean))
        return pred.clip(lower=0).values


run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='PatchTST_Final_Pipeline',
                 job_type='final_pipeline',
                 config={**BEST_CFG, 'pred_len': PRED_LEN_TEST, 'exogenous': 'IsHoliday'})
run.summary['val_wmae'] = best_val

mu_f = wide.values.mean(axis=0)
sigma_f = np.clip(wide.values.std(axis=0), 1.0, None)
norm_full = (wide.values - mu_f) / sigma_f
full_holiday = holiday_by_date.reindex(wide.index).values

final_model, losses = train_patchtst(norm_full, full_holiday, pred_len=PRED_LEN_TEST, **BEST_CFG,
                                      log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))

pred = forecast_next(final_model, norm_full, test_holiday_future, BEST_CFG['seq_len']) * sigma_f + mu_f
pred = np.clip(pred, 0, None)

pred_df = pd.DataFrame(pred, index=test_dates, columns=wide.columns)
forecast_long = pred_df.stack([0, 1]).rename('pred').reset_index()
forecast_long.columns = ['Date', 'Store', 'Dept', 'pred']

sd_mean = train.groupby(['Store', 'Dept'])['Weekly_Sales'].mean().rename('SD_Mean').reset_index()
dept_mean = train.groupby('Dept')['Weekly_Sales'].mean().rename('Dept_Mean').reset_index()
global_mean = float(train['Weekly_Sales'].mean())

pipeline = PatchTSTForecastPipeline(forecast_long, sd_mean, dept_mean, global_mean)
val_score = best_val

with open('patchtst_pipeline.pkl', 'wb') as f:
    cloudpickle.dump(pipeline, f)
art = wandb.Artifact('patchtst_pipeline', type='model')
art.add_file('patchtst_pipeline.pkl')
run.log_artifact(art)
run.finish()
print('Pipeline saved. val_wmae:', val_score)

test horizon: 39 weeks


  epoch  10: train MSE 0.6226
  epoch  20: train MSE 0.5843
  epoch  30: train MSE 0.5630
  epoch  40: train MSE 0.5491
  epoch  50: train MSE 0.5398
  epoch  60: train MSE 0.5355
  epoch  70: train MSE 0.5286
  epoch  80: train MSE 0.5234


/tmp/ipykernel_3460/1850727755.py:46: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  forecast_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇██
train_mse,█▅▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,79
train_mse,0.52344
val_wmae,1499.25118


Pipeline saved. val_wmae: 1499.25118


In [12]:
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
info = json.load(open(reg_path)) if os.path.exists(reg_path) else {}
info['PatchTST'] = {
    'artifact': f'{WANDB_PROJECT}/patchtst_pipeline:latest',
    'val_wmae': float(val_score),
}
with open(reg_path, 'w') as f:
    json.dump(info, f, indent=2)
info

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest',
  'val_wmae': 1451.361784955096},
 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest',
  'val_wmae': 1508.4106169389534},
 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest',
  'val_wmae': 1602.6865936618267},
 'NBEATS': {'artifact': 'ML-Final/nbeats_pipeline:latest',
  'val_wmae': 1463.00219},
 'PatchTST': {'artifact': 'ML-Final/patchtst_pipeline:latest',
  'val_wmae': 1499.25118},
 'Prophet': {'artifact': 'ML-Final/prophet_pipeline:latest',
  'val_wmae': 1831.404616769137},
 'ARIMA': {'artifact': 'ML-Final/arima_pipeline:latest',
  'val_wmae': 1914.1948218719908},
 'SARIMA': {'artifact': 'ML-Final/sarima_pipeline:latest',
  'val_wmae': 2137.2593427953866}}

In [ ]:
api = wandb.Api()
art = api.artifact(f'{WANDB_PROJECT}/patchtst_pipeline:latest')
art_dir = art.download()
with open(os.path.join(art_dir, 'patchtst_pipeline.pkl'), 'rb') as f:
    loaded = cloudpickle.load(f)
print(loaded.predict(test[RAW_COLS].head())[:5])

wandb:   1 of 1 files downloaded.  


[32866.0198363  21720.97909876 18752.60834171 19952.0190157
 23290.91848593]


Experimental Canditates

In [9]:
tr_wide = wide[wide.index < VAL_CUTOFF]
mu = tr_wide.values.mean(axis=0)
sigma = np.clip(tr_wide.values.std(axis=0), 1.0, None)
norm_tr = (tr_wide.values - mu) / sigma

tr_holiday = holiday_by_date.reindex(tr_wide.index).values
val_holiday_future = holiday_by_date.reindex(val_dates).values.astype(np.float32)

best_val = 1554.19
BEST_CFG = dict(seq_len=52, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3)

EXTRA_CANDIDATES = [
    dict(seq_len=52, patch_len=16, stride=8, d_model=64, n_heads=8, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=16, stride=8, d_model=128, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=16, stride=8, d_model=64, n_heads=4, n_layers=3, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=16, stride=8, d_model=64, n_heads=4, n_layers=1, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=24, stride=12, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=12, stride=6, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=104, patch_len=16, stride=8, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
    dict(seq_len=52, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=2e-3),
    dict(seq_len=52, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, epochs=120, lr=1e-3),
    dict(seq_len=52, patch_len=8, stride=4, d_model=32, n_heads=2, n_layers=1, epochs=80, lr=1e-3),
    dict(seq_len=78, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=1e-3),
]

for j, cfg in enumerate(EXTRA_CANDIDATES):
    i = 4 + j
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'PatchTST_candidate_{i}',
                     job_type='hyperparam_search', reinit=True,
                     config={**cfg, 'pred_len': PRED_LEN_VAL, 'exogenous': 'IsHoliday',
                             'val_scheme': f'train < {VAL_CUTOFF}, predict {PRED_LEN_VAL} weeks ahead',
                             'sweep_phase': 'extended'})
    print(f'candidate {i}: {cfg}')
    model, losses = train_patchtst(norm_tr, tr_holiday, pred_len=PRED_LEN_VAL, **cfg,
                                    log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))
    pred = forecast_next(model, norm_tr, val_holiday_future, cfg['seq_len']) * sigma + mu
    score = float(matrix_val_wmae(pred))
    run.summary['val_wmae'] = score
    run.finish()
    print(f'  -> val WMAE {score:,.2f}')
    if score < best_val:
        best_val, BEST_CFG = score, cfg

print('\nOverall best after extended search:', BEST_CFG, f'-> {best_val:,.2f}')

epoch,▁█
train_mse,█▁
epoch,1
train_mse,0.87255


candidate 4: {'seq_len': 52, 'patch_len': 16, 'stride': 8, 'd_model': 64, 'n_heads': 8, 'n_layers': 2, 'epochs': 80, 'lr': 0.001}
  epoch  10: train MSE 0.6728
  epoch  20: train MSE 0.6421
  epoch  30: train MSE 0.6288
  epoch  40: train MSE 0.6190
  epoch  50: train MSE 0.6099
  epoch  60: train MSE 0.6011
  epoch  70: train MSE 0.5939
  epoch  80: train MSE 0.5877


/tmp/ipykernel_3460/1708944223.py:64: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train_mse,█▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,79
train_mse,0.58771
val_wmae,1729.40914


  -> val WMAE 1,729.41


...

  -> val WMAE 1,594.36


candidate 14: {'seq_len': 78, 'patch_len': 8, 'stride': 4, 'd_model': 64, 'n_heads': 4, 'n_layers': 2, 'epochs': 80, 'lr': 0.001}
  epoch  10: train MSE 0.6635
  epoch  20: train MSE 0.6071
  epoch  30: train MSE 0.5849
  epoch  40: train MSE 0.5668
  epoch  50: train MSE 0.5566
  epoch  60: train MSE 0.5476
  epoch  70: train MSE 0.5387
  epoch  80: train MSE 0.5351


/tmp/ipykernel_3460/1708944223.py:64: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pred_long = pred_df.stack([0, 1]).rename('pred').reset_index()


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
train_mse,█▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,79
train_mse,0.53515
val_wmae,1577.07654


  -> val WMAE 1,577.08

Overall best after extended search: {'seq_len': 52, 'patch_len': 8, 'stride': 4, 'd_model': 64, 'n_heads': 4, 'n_layers': 2, 'epochs': 80, 'lr': 0.002} -> 1,499.25


In [10]:
best_val = 1499.25118
BEST_CFG = dict(seq_len=52, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, epochs=80, lr=2e-3)